# C, D. 시장성 · 이해관계자 평가 에이전트

| | C. 시장성 | D. 이해관계자 |
|---|---|---|
| **담당** | Web Search | Web Search |
| **선행 노드** | B | B |
| **출력** | `market_eval`, `market_references` | `stakeholder_eval`, `stakeholder_references` |

둘 다 RAG 없이 웹 검색만 쓰고, `tech_research`(B의 결과)를 참고해 검색어를 구체화한다는 점이 같아서 한 노트북에 같이 둔다.
시장성은 "문서·저장소·규격의 기록"만, 이해관계자는 "발화"만 본다 — 겹치지 않게 역할이 나뉜다(2-2/2-3절).

이 노트북 끝에서 만든 함수 둘은 `src/nodes_cd.py`로 저장된다.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import prompts
from src.node_utils import run_web_search, summarize_tech_research
from src.schemas import MarketEval, StakeholderEval

## 1. 프롬프트 확인

In [ ]:
print(prompts.MARKET_EVAL_PROMPT)
print("\n" + "="*80 + "\n")
print(prompts.STAKEHOLDER_EVAL_PROMPT)

## 2. 노드 함수 정의

In [ ]:
def split_web_references(search_results, queries):
    """run_web_search 가 합쳐 준 문자열을 쿼리 단위로 되쪼개 Reference 목록을 만든다.

    이전 판은 검색 결과 전체를 200자로 잘라 {"source": "web_search"} 한 건으로
    뭉쳤다. 그러면 REFERENCE 장에 URL 이 한 줄도 안 남는다. 여기서는
    쿼리마다 한 건으로 나누고 본문에서 URL 을 뽑아 함께 싣는다.
    """
    import re

    blocks = {}
    current = None
    for line in search_results.split("\n"):
        m = re.match(r"^\[(.+?)\]\s?(.*)$", line)
        if m and m.group(1) in queries:
            current = m.group(1)
            blocks[current] = [m.group(2)]
        elif current is not None:
            blocks[current].append(line)

    refs = []
    for q in queries:
        body = "\n".join(blocks.get(q, []))
        urls = [
            u.rstrip(".,;:)}'\"")
            for u in re.findall(r"https?://\S+", body)
        ]
        seen = []
        for u in urls:
            if u not in seen:
                seen.append(u)
        refs.append(
            {
                "source": f"web_search: {q}",
                "detail": (
                    ("URL: " + " | ".join(seen[:3]) + " / ") if seen else "URL 없음 / "
                )
                + body.strip()[:300],
            }
        )
    return refs


def make_node_c(llm, web_search_tool):
    """C. 시장성 평가."""
    structured_llm = llm.with_structured_output(MarketEval)

    # 대칭 질의: 템플릿 1벌을 두 기술에 기술명만 바꿔 던진다.
    QUERY_TEMPLATES = [
        "{tech} KV cache adoption production release notes 2024 2025 2026",
        "{tech} KV cache support merged llama.cpp MLX vLLM GitHub",
        "{tech} on-device edge deployment smartphone integration",
    ]

    def node_c_market_eval(state):
        tech_context = summarize_tech_research(state.get("tech_research", {}))
        queries = [
            t.format(tech=tech)
            for tech in ("TurboQuant", "InfiniGen")
            for t in QUERY_TEMPLATES
        ]
        search_results = run_web_search(web_search_tool, queries)
        prompt = prompts.MARKET_EVAL_PROMPT.format(
            tech_context=tech_context, search_results=search_results
        )
        result = structured_llm.invoke(prompt)
        return {
            "market_eval": result.model_dump(),
            "market_references": split_web_references(search_results, queries),
        }

    return node_c_market_eval

In [ ]:
def make_node_d(llm, web_search_tool):
    """D. 이해관계자 평가."""
    structured_llm = llm.with_structured_output(StakeholderEval)

    # 대칭 질의 + 찬반 대칭: 기술마다 지지/비판/투자 3방향으로 검색.
    QUERY_TEMPLATES = [
        "{tech} KV cache developer feedback adoption benchmark results",
        "{tech} KV cache criticism limitation concern drawback",
        "{tech} KV cache investor analyst report coverage media 2024 2025 2026",
        "{tech} vs competing KV cache method comparison response",
    ]

    def node_d_stakeholder_eval(state):
        tech_context = summarize_tech_research(state.get("tech_research", {}))
        queries = [
            t.format(tech=tech)
            for tech in ("TurboQuant", "InfiniGen")
            for t in QUERY_TEMPLATES
        ]
        search_results = run_web_search(web_search_tool, queries)
        prompt = prompts.STAKEHOLDER_EVAL_PROMPT.format(
            tech_context=tech_context, search_results=search_results
        )
        result = structured_llm.invoke(prompt)
        return {
            "stakeholder_eval": result.model_dump(),
            "stakeholder_references": split_web_references(search_results, queries),
        }

    return node_d_stakeholder_eval

## 3. 배선 테스트 — API 키 없이

In [ ]:
from src.schemas import TechStatus

class FakeStructuredLLM:
    def __init__(self, output):
        self.output = output
    def invoke(self, prompt):
        return self.output

# TechStatus 는 제네릭이라 필드마다 허용 값이 다르다(schemas.py).
# 한 벌을 돌려 쓰면 ValidationError 가 난다 - 필드별로 맞는 값을 넣는다.
class FakeLLM_C:
    def with_structured_output(self, schema_cls):
        return FakeStructuredLLM(MarketEval(
            market_size_growth="추정 갈림",
            adoption_status=TechStatus(turboquant="정식", infinigen="실험"),
            ecosystem_support=TechStatus(turboquant="본류 병합", infinigen="포크만"),
            standardization="있음", label="조건 의존", notes="가짜",
        ))

class FakeLLM_D:
    def with_structured_output(self, schema_cls):
        return FakeStructuredLLM(StakeholderEval(
            competing_camp_reaction=TechStatus(turboquant="한계 지적", infinigen="언급만"),
            developer_adoption=TechStatus(turboquant="채택했다고 말함", infinigen="조건부"),
            investor_coverage=TechStatus(turboquant="있음", infinigen="근거 없음"),
            label="조건 의존", notes="가짜",
        ))

class FakeWebSearchTool:
    def invoke(self, args):
        return f"가짜 검색 결과: {args['query']}"

sample_state = {"tech_research": {"TurboQuant": {"overview": "가짜 개요"}}}

node_c = make_node_c(FakeLLM_C(), FakeWebSearchTool())
result_c = node_c(sample_state)
assert result_c["market_eval"]["label"] == "조건 의존"
# 레퍼런스는 쿼리 1개당 1건. 대칭 질의라 2기술 x 템플릿 수가 그대로 개수가 된다.
# 이 단언이 깨지면 QUERY_TEMPLATES 가 바뀐 것이다 - 의도한 변경인지 먼저 확인할 것.
assert len(result_c["market_references"]) == 6, len(result_c["market_references"])
print("C 배선 OK:", result_c["market_eval"]["adoption_status"],
      "| 레퍼런스", len(result_c["market_references"]), "건")

node_d = make_node_d(FakeLLM_D(), FakeWebSearchTool())
result_d = node_d(sample_state)
assert result_d["stakeholder_eval"]["label"] == "조건 의존"
assert len(result_d["stakeholder_references"]) == 8, len(result_d["stakeholder_references"])
print("D 배선 OK:", result_d["stakeholder_eval"]["developer_adoption"],
      "| 레퍼런스", len(result_d["stakeholder_references"]), "건")

## 4. 실제 LLM 테스트

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

if os.environ.get("OPENAI_API_KEY") and os.environ.get("TAVILY_API_KEY"):
    from langchain.chat_models import init_chat_model
    from langchain_tavily import TavilySearch
    from src import config

    real_llm = init_chat_model(config.LLM_MODEL, model_provider=config.LLM_PROVIDER, temperature=0)
    real_web_search = TavilySearch(max_results=5)

    node_c_real = make_node_c(real_llm, real_web_search)
    print(node_c_real(sample_state)["market_eval"])
else:
    print("API 키 없음 - 이 셀은 건너뜀.")

## 5. 파일로 저장

In [ ]:
import inspect

TARGET = "../src/nodes_cd.py"

# 심볼 유실 감지. 아래 parts 목록은 하드코딩이라, 누가 src/nodes_cd.py 을 직접 고쳐
# 함수·상수를 더해 놓으면 저장하는 순간 그게 조용히 사라진다(2026-09-22 실제 발생).
# 사라진 이름이 있으면 여기서 알린다 - 에러가 안 나서 안 보이는 게 진짜 위험이다.
def _symbols(path):
    import ast, os
    if not os.path.exists(path):
        return set()
    out = set()
    for n in ast.parse(open(path, encoding="utf-8").read()).body:
        if isinstance(n, ast.FunctionDef):
            out.add(n.name)
        elif isinstance(n, ast.Assign):
            out |= {t.id for t in n.targets if isinstance(t, ast.Name)}
    return out


def _src(obj):
    """getsource 결과의 꼬리 개행을 없앤다. 셀 마지막에 있는 함수는 개행이
    안 붙어 나와서, 그대로 이으면 최상위 정의 사이가 빈 줄 1개가 된다."""
    return inspect.getsource(obj).rstrip("\n")


_before = _symbols(TARGET)

q3 = chr(34) * 3
HEADER = (
    q3 + "C, D. 시장성/이해관계자 노드 - 02_agent_CD_market_stakeholder.ipynb에서 생성됨.\n"
    "이 파일을 직접 고치지 말고, 노트북에서 고친 뒤 저장 셀을 다시 실행할 것." + q3 + "\n\n"
    "from src import prompts\n"
    "from src.node_utils import run_web_search, summarize_tech_research\n"
    "from src.schemas import MarketEval, StakeholderEval\n\n\n"
)

parts = [
    _src(split_web_references),
    _src(make_node_c),
    _src(make_node_d),
]

with open(TARGET, "w", encoding="utf-8") as f:
    f.write(HEADER + "\n\n\n".join(parts) + "\n")   # 정의 사이는 빈 줄 2개(PEP8)

_lost = sorted(_before - _symbols(TARGET))
if _lost:
    print(f"🔴 이번 저장으로 {TARGET} 에서 사라진 심볼: {_lost}")
    print("   노트북이 .py 보다 낡았다는 뜻이다. git diff 로 확인하고,")
    print("   의도한 삭제가 아니면 git checkout 으로 되돌린 뒤 노트북부터 맞출 것.")
else:
    print(f"{TARGET} 저장 완료 (심볼 유실 없음)")